# Voice-to-Voice Agent with Eden AI

A push-to-talk voice agent that chains three Eden AI features through one API key:

**Speech-to-Text → LLM → Text-to-Speech**

Click record, speak, click stop, then send. The audio is uploaded, transcribed, sent to an LLM with conversation history, and the answer is spoken back. Swap any provider in one line at the top of the config cell.

**Prerequisites:** an Eden AI API key (set as `EDENAI_API_KEY` env var), a microphone, and a browser-based Jupyter frontend (Classic Notebook or JupyterLab — VS Code's Jupyter extension does not expose mic permissions reliably).

In [1]:
%pip install --quiet requests ipywebrtc ipywidgets python-dotenv

Note: you may need to restart the kernel to use updated packages.


## 1. Configuration

All provider/model strings live here — swap them to compare providers without touching the rest of the notebook.

In [2]:
import base64
import json
import os

from dotenv import load_dotenv
from IPython.display import HTML, display

load_dotenv(override=True)

EDENAI_API_KEY = os.environ.get("EDENAI_API_KEY")
if not EDENAI_API_KEY:
    raise RuntimeError("Set EDENAI_API_KEY (env var or .env file). Get one at https://app.edenai.run")
EDENAI_BASE = "https://api.edenai.run/v3"

STT_MODEL = "audio/speech_to_text_async/openai"
LLM_MODEL = "anthropic/claude-sonnet-4-5"
TTS_MODEL = "audio/tts/openai/tts-1"

LANGUAGE = "en"
SYSTEM_PROMPT = "You are a concise voice assistant. Keep replies under two short sentences."

HEADERS = {"Authorization": f"Bearer {EDENAI_API_KEY}"}
JSON_HEADERS = {**HEADERS, "Content-Type": "application/json"}


def _is_sandbox(jwt: str) -> bool:
    try:
        payload_b64 = jwt.split(".")[1]
        payload_b64 += "=" * (-len(payload_b64) % 4)
        return json.loads(base64.urlsafe_b64decode(payload_b64)).get("type") == "sandbox_api_token"
    except Exception:
        return False


if _is_sandbox(EDENAI_API_KEY):
    display(HTML(
        '<div style="background:#fff3cd;border-left:4px solid #ffc107;padding:10px 14px;'
        'border-radius:4px;font-family:sans-serif;font-size:13px;margin:6px 0;">'
        '<b>⚠ Sandbox key detected.</b> Every model returns the same mocked text and TTS audio '
        'is too small to round-trip through upload + STT. '
        'Use a production key from <a href="https://app.edenai.run" target="_blank">app.edenai.run</a> '
        'to run the real pipeline.</div>'
    ))

## 2. Eden AI helpers

Four small functions wrap the three Eden AI endpoints used by the pipeline:

- `upload_file` → `POST /v3/upload` (multipart) — returns a `file_id` valid for 7 days
- `transcribe` → `POST /v3/universal-ai/async` then poll `GET /v3/universal-ai/async/{job_id}`
- `chat` → `POST /v3/llm/chat/completions` (OpenAI-compatible)
- `synthesize` → `POST /v3/universal-ai` (sync) returns an `audio_resource_url` we then download

In [3]:
import time
import requests


def upload_file(audio_bytes: bytes, filename: str = "recording.webm") -> str:
    files = {"file": (filename, audio_bytes, "audio/webm")}
    data = {"purpose": "speech"}
    r = requests.post(f"{EDENAI_BASE}/upload", headers=HEADERS, files=files, data=data)
    r.raise_for_status()
    return r.json()["file_id"]


def transcribe(file_id: str, language: str = LANGUAGE) -> str:
    payload = {
        "model": STT_MODEL,
        "input": {"file": file_id, "language": language},
    }
    r = requests.post(f"{EDENAI_BASE}/universal-ai/async", headers=JSON_HEADERS, json=payload)
    r.raise_for_status()
    data = r.json()

    # Short clips often return the result inline on the launch call.
    if data.get("status") == "success":
        return data["output"]["text"]

    job_id = data["public_id"]
    while True:
        time.sleep(1)
        s = requests.get(f"{EDENAI_BASE}/universal-ai/async/{job_id}", headers=HEADERS)
        s.raise_for_status()
        data = s.json()
        status = data.get("status")
        if status == "success":
            return data["output"]["text"]
        if status == "failed":
            raise RuntimeError(data.get("error") or "STT job failed")


def chat(messages: list) -> str:
    payload = {"model": LLM_MODEL, "messages": messages}
    r = requests.post(f"{EDENAI_BASE}/llm/chat/completions", headers=JSON_HEADERS, json=payload)
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"]


def synthesize(text: str) -> bytes:
    payload = {"model": TTS_MODEL, "input": {"text": text}}
    r = requests.post(f"{EDENAI_BASE}/universal-ai", headers=JSON_HEADERS, json=payload)
    r.raise_for_status()
    audio_url = r.json()["output"]["audio_resource_url"]
    return requests.get(audio_url).content

## 3. Push-to-talk widget

`ipywebrtc.AudioRecorder` gives us in-browser mic capture. Click ● to record, ■ to stop, then **Send to agent**.

Conversation history accumulates across rounds, so the agent remembers what you said before.

In [4]:
import time

from ipywebrtc import AudioRecorder, CameraStream
from ipywidgets import Button, HBox, HTML as HTMLWidget, Output, VBox
from IPython.display import Audio, HTML, clear_output, display

stream = CameraStream(constraints={"audio": True, "video": False})
recorder = AudioRecorder(stream=stream)

send_btn = Button(description="Send to agent", button_style="primary")
reset_btn = Button(description="Reset conversation")


def _pill(text, color):
    return (
        f'<span style="background:{color};color:white;padding:4px 12px;border-radius:12px;'
        f'font-size:12px;font-family:sans-serif;font-weight:600;">{text}</span>'
    )


status_pill = HTMLWidget(value=_pill("idle", "#6c757d"))
transcript_out = Output()
audio_out = Output()
errors_out = Output()

history = [{"role": "system", "content": SYSTEM_PROMPT}]
turns = []


def _set_status(text, color):
    status_pill.value = _pill(text, color)


def _render_transcript():
    with transcript_out:
        clear_output()
        for turn in turns:
            user_bubble = (
                f'<div style="background:#e3f2fd;padding:8px 12px;border-radius:12px;'
                f'margin:6px 0;max-width:75%;font-family:sans-serif;font-size:13px;">'
                f'<b>🗣 You</b><br>{turn["user"]}</div>'
            )
            bot_bubble = (
                f'<div style="background:#f3e5f5;padding:8px 12px;border-radius:12px;'
                f'margin:6px 0 6px auto;max-width:75%;font-family:sans-serif;font-size:13px;">'
                f'<b>🤖 Agent</b><br>{turn["assistant"]}</div>'
            )
            timings = (
                f'<div style="font-size:11px;color:#888;font-family:sans-serif;'
                f'margin-bottom:10px;text-align:right;">'
                f'STT {turn["stt_ms"]:.0f}ms · LLM {turn["llm_ms"]:.0f}ms · TTS {turn["tts_ms"]:.0f}ms</div>'
            )
            display(HTML(user_bubble + bot_bubble + timings))


def run_pipeline(audio_bytes: bytes) -> None:
    errors_out.clear_output()
    try:
        _set_status("uploading…", "#ffc107")
        file_id = upload_file(audio_bytes)

        _set_status("transcribing…", "#17a2b8")
        t = time.perf_counter()
        user_text = transcribe(file_id)
        stt_ms = (time.perf_counter() - t) * 1000

        history.append({"role": "user", "content": user_text})

        _set_status("thinking…", "#007bff")
        t = time.perf_counter()
        reply = chat(history)
        llm_ms = (time.perf_counter() - t) * 1000
        history.append({"role": "assistant", "content": reply})

        _set_status("synthesizing…", "#6610f2")
        t = time.perf_counter()
        audio = synthesize(reply)
        tts_ms = (time.perf_counter() - t) * 1000

        turns.append({
            "user": user_text, "assistant": reply,
            "stt_ms": stt_ms, "llm_ms": llm_ms, "tts_ms": tts_ms,
        })
        _render_transcript()

        with audio_out:
            clear_output(wait=True)
            display(Audio(audio, autoplay=True))

        _set_status("idle", "#6c757d")
    except Exception as e:
        _set_status("error ✗", "#dc3545")
        with errors_out:
            display(HTML(
                f'<div style="color:#dc3545;font-family:monospace;font-size:12px;'
                f'padding:6px 10px;background:#f8d7da;border-radius:4px;">Error: {e}</div>'
            ))


def on_send(_):
    audio_bytes = recorder.audio.value
    if not audio_bytes:
        with errors_out:
            clear_output()
            display(HTML(
                '<div style="color:#666;font-family:sans-serif;font-size:12px;">'
                '(record something first)</div>'
            ))
        return
    run_pipeline(audio_bytes)


def on_reset(_):
    history.clear()
    history.append({"role": "system", "content": SYSTEM_PROMPT})
    turns.clear()
    transcript_out.clear_output()
    errors_out.clear_output()
    audio_out.clear_output()
    _set_status("idle", "#6c757d")


send_btn.on_click(on_send)
reset_btn.on_click(on_reset)

controls = HBox([send_btn, reset_btn, status_pill])
display(VBox([recorder, controls, transcript_out, audio_out, errors_out]))

## 4. Try swapping providers

Edit the constants at the top, re-run the config cell, and keep talking. Some combinations to try:

| Goal | Suggested model |
|---|---|
| Cheap STT | `audio/speech_to_text_async/deepgram/nova-3` |
| Higher voice quality | `audio/tts/elevenlabs/eleven_turbo_v2_5` |
| Cheap multilingual TTS | `audio/tts/google/wavenet` |
| Different reasoning | `openai/gpt-4`, `deepseek/deepseek-chat`, `google/gemini-2.5-flash` |

## 5. Variant: voice translator

Replace the LLM step in `run_pipeline` with a translation prompt and pick a multilingual TTS:

```python
TTS_MODEL = "audio/tts/elevenlabs/eleven_multilingual_v2"

def translate(text: str, target: str = "French") -> str:
    return chat([
        {"role": "system", "content": f"Translate the user's text to {target}. Output only the translation."},
        {"role": "user", "content": text},
    ])

# in run_pipeline, replace `reply = chat(history)` with:
# reply = translate(user_text, target="French")
```

Same three-feature pipeline, completely different application.